# osu-skill-predictor — Project Backlog

This notebook contains the implementation backlog for a production-shaped classical ML API that predicts osu! beatmap pass probability and expected accuracy.

## Project Overview

### What the project does

`osu-skill-predictor` is a classical ML service that estimates whether a player can pass a given osu! beatmap and predicts the player's likely accuracy on that map. The long-term shape of the project is a small production-style ML system with local training, serialized model artifacts, a FastAPI inference service, tests, and lightweight documentation.

### Why the project exists

The project is designed as a first portfolio ML project for the issue `BL-009 Project 1: tabular ML prediction API`. It should bridge backend engineering experience with ML fundamentals by focusing on a narrow, realistic prediction task that can be implemented with tabular features and classical models.

### Why osu! is a good learning domain

- The domain is personally motivating, which increases the chance of finishing the MVP.
- Beatmaps and player summaries can be represented as structured tabular features.
- Difficulty concepts such as star rating, BPM, AR, OD, and mods are intuitive enough to reason about while learning feature engineering.
- The problem naturally supports both classification and regression in one product.

### What the MVP includes

- A clearly defined local dataset schema.
- A small initial tabular dataset for experimentation.
- A pass/fail classifier.
- An accuracy regressor.
- Serialized scikit-learn model artifacts.
- A FastAPI `/predict` endpoint plus a `/health` endpoint.
- Basic automated tests.
- A short model card and documentation of assumptions and limitations.

### What is out of scope for v1

- Deep learning or sequence-based models.
- Online training or continuous retraining.
- Real-time gameplay event ingestion.
- Ranking-grade recommendation systems.
- Complex personalization beyond a compact tabular feature set.
- Production deployment infrastructure.

## Product Goal

### Main user story

As an osu! player, I want to submit a summary of my recent skill profile together with beatmap characteristics so that I can estimate whether the map is likely passable for me and what accuracy range I should expect.

### Example API input

```json
{
  "user_avg_star_rating": 5.2,
  "user_avg_accuracy": 96.4,
  "user_avg_pp": 180,
  "beatmap_star_rating": 5.6,
  "beatmap_bpm": 185,
  "beatmap_ar": 9.3,
  "beatmap_od": 8.7,
  "beatmap_cs": 4.0,
  "beatmap_length_sec": 132,
  "mods": "HDHR"
}
```

### Example API output

```json
{
  "pass_probability": 0.78,
  "predicted_accuracy": 94.2,
  "difficulty_gap": 0.4,
  "recommendation": "Playable, but slightly above comfort zone"
}
```

### Product notes

- The first version should prioritize clarity and correctness over ambitious modeling.
- The API output should remain interpretable for a human player.
- Recommendation text can start as a rule-based layer derived from model outputs.

## ML Problem Definition

### Primary targets

- Classification target: `pass` / `fail`.
- Regression target: predicted accuracy as a continuous percentage value.

### Modeling shape for MVP

- Train separate classical models for classification and regression.
- Prefer scikit-learn estimators and pipelines.
- Keep feature engineering explicit and explainable.

### Possible future targets

- FC probability.
- PP prediction.
- Recommendation score.
- Map comfort score.

### Constraints

- Use classical ML only for v1.
- Do not introduce deep learning in the MVP.
- Optimize for beginner-friendly implementation that still looks portfolio-worthy.

## Data Backlog

- [x] P0 / M — Create initial dataset schema
  - Description: Define the columns required for the first training dataset for both input features and targets.
  - Acceptance criteria:
    - `docs/dataset_schema.md` exists
    - Required input features are documented
    - Classification and regression targets are documented

- [x] P0 / M — Decide the first record granularity
  - Description: Choose what one row means, such as one player-beatmap attempt or one aggregated player-map sample.
  - Acceptance criteria:
    - A single row definition is documented
    - The target labels are compatible with that row definition
    - Tradeoffs are noted in the notebook or docs

- [x] P0 / M — Collect or create a small local starter dataset
  - Description: Build a small sample dataset that is enough to exercise the pipeline and API contract locally.
  - Acceptance criteria:
    - A starter dataset exists under `data/raw/` or `data/sample/`
    - The file can be loaded locally without external services
    - The dataset contains both targets or a plan for deriving them

- [x] P0 / S — Document the data source and provenance
  - Description: Record where the data came from, what assumptions were made, and any synthetic or manually curated elements.
  - Acceptance criteria:
    - Data source notes are written
    - Manual transformations are listed
    - Limitations of the starter dataset are documented

- [x] P0 / M — Validate required columns and data types
  - Description: Define basic validation expectations before model training begins.
  - Acceptance criteria:
    - Required columns are enumerated
    - Expected dtypes or value domains are listed
    - A validation checklist exists for the raw dataset

- [x] P1 / M — Define missing-value handling rules
  - Description: Decide which columns can be imputed, defaulted, or rejected.
  - Acceptance criteria:
    - Missing-value policy is documented per feature group
    - Drop vs impute decisions are explained
    - Invalid-row handling is specified

- [x] P0 / S — Plan the train/test split strategy
  - Description: Choose a split approach that is simple, reproducible, and appropriate for the initial dataset size.
  - Acceptance criteria:
    - Split ratio is documented
    - Random seed strategy is documented
    - Leakage risks are noted

- [x] P1 / S — Define processed dataset outputs
  - Description: Decide what cleaned or transformed datasets should be saved for later steps.
  - Acceptance criteria:
    - `data/processed/` output files are identified
    - Save format is chosen
    - Each saved artifact has a clear purpose


## Feature Engineering Backlog

- [x] P0 / M — Define the MVP beatmap feature set
  - Description: Finalize which beatmap attributes are required for v1 predictions.
  - Acceptance criteria:
    - Core beatmap features are listed
    - Units and valid ranges are documented
    - Any excluded candidate features are noted

- [x] P0 / M — Define the MVP player feature set
  - Description: Finalize which player summary attributes represent current skill in a tabular way.
  - Acceptance criteria:
    - Core player features are listed
    - Aggregation windows or assumptions are documented
    - Each feature has a rationale

- [x] P0 / S — Define mod parsing rules
  - Description: Decide how to represent mods consistently from raw string inputs.
  - Acceptance criteria:
    - Supported mods for v1 are listed
    - Combined mod string parsing rules are defined
    - Unknown or unsupported mods have a documented behavior

- [x] P0 / M — Add engineered difficulty-gap features
  - Description: Create explicit gap-style features that compare player comfort level to beatmap difficulty.
  - Acceptance criteria:
    - `star_gap` is defined
    - `bpm_gap` is defined if player BPM proxy exists
    - `od_gap` and `ar_gap` are defined or explicitly deferred

- [x] P1 / S — Add bucketed and binary helper features
  - Description: Introduce simple interpretable features that may improve baseline performance.
  - Acceptance criteria:
    - `length_bucket` is defined
    - `has_hidden` is defined
    - `has_hardrock` is defined
    - `has_doubletime` is defined

- [x] P1 / S — Choose categorical encoding strategy
  - Description: Decide how categorical or mod-derived fields will enter the pipeline.
  - Acceptance criteria:
    - Encoding approach is documented
    - The chosen approach is compatible with scikit-learn pipelines
    - High-cardinality risks are considered if relevant

- [x] P1 / S — Decide whether numeric scaling is needed
  - Description: Evaluate whether scaling should be part of the default preprocessing path for candidate models.
  - Acceptance criteria:
    - A scaling decision is documented
    - The decision references likely baseline estimators
    - Pipeline implications are noted


## Model Training Backlog

- [x] P0 / M — Select a baseline classifier
  - Description: Start with one simple scikit-learn classification model that is easy to explain and evaluate.
  - Acceptance criteria:
    - A baseline classifier candidate is chosen
    - The choice is justified briefly
    - Inputs and targets are clearly mapped

- [x] P0 / M — Select a baseline regressor
  - Description: Start with one simple scikit-learn regression model for predicted accuracy.
  - Acceptance criteria:
    - A baseline regressor candidate is chosen
    - The choice is justified briefly
    - Expected output scale is documented

- [x] P0 / M — Design a shared scikit-learn preprocessing pipeline
  - Description: Use a pipeline or column transformer design that keeps preprocessing reproducible.
  - Acceptance criteria:
    - Preprocessing steps are listed
    - The design supports local training and inference reuse
    - The plan is compatible with joblib serialization

- [x] P0 / S — Define evaluation metrics
  - Description: Pick metrics that are understandable and sufficient for a first portfolio-quality iteration.
  - Acceptance criteria:
    - Classification metrics are listed
    - Regression metrics are listed
    - A primary metric is chosen for each model

- [x] P1 / M — Plan a small model comparison pass
  - Description: Compare a few reasonable classical models without turning the project into tuning-heavy research.
  - Acceptance criteria:
    - A short candidate model list exists
    - Comparison criteria are documented
    - Overfitting and dataset-size caveats are noted

- [x] P0 / S — Define serialized artifact outputs
  - Description: Decide what exactly gets saved after training and how those artifacts are named.
  - Acceptance criteria:
    - `models/pass_model.joblib` is planned
    - `models/accuracy_model.joblib` is planned
    - Metadata or versioning notes are documented if needed


## API Backlog

- [x] P0 / M — Create FastAPI application skeleton
  - Description: Define the minimal application layout for local inference serving.
  - Acceptance criteria:
    - App module responsibilities are mapped
    - Entry point location is decided
    - Local startup approach is documented

- [x] P0 / S — Define request schema
  - Description: Convert the planned feature contract into a validated API input model.
  - Acceptance criteria:
    - Required request fields are listed
    - Field types are specified
    - Input validation expectations are documented

- [x] P0 / S — Define response schema
  - Description: Formalize the initial API response structure for predictions and recommendation text.
  - Acceptance criteria:
    - Response fields are listed
    - Numeric outputs are defined with expected ranges or meanings
    - Recommendation text behavior is documented

- [x] P0 / S — Add `/health` endpoint requirements
  - Description: Specify what a healthy service means in the MVP.
  - Acceptance criteria:
    - Health response payload is described
    - Model-loading dependency behavior is defined
    - Failure mode expectations are noted

- [x] P0 / M — Add `/predict` endpoint requirements
  - Description: Specify the end-to-end inference contract for combined classifier and regressor predictions.
  - Acceptance criteria:
    - Request and response mapping is documented
    - The order of feature preparation and prediction is defined
    - Error responses are considered

- [x] P0 / S — Plan model artifact loading behavior
  - Description: Decide when models are loaded and how inference code accesses them.
  - Acceptance criteria:
    - Startup-time vs lazy-loading decision is documented
    - Missing-model behavior is documented
    - Artifact paths are explicit

- [x] P1 / S — Define error handling rules
  - Description: Document how the API should respond to invalid inputs and internal prediction failures.
  - Acceptance criteria:
    - Validation error shape is described
    - Internal error handling policy is described
    - The service does not expose confusing raw stack-trace details

- [x] P1 / S — Write local run instructions backlog item
  - Description: Capture the minimum instructions needed to train locally and run the API.
  - Acceptance criteria:
    - Required commands are identified
    - Dependency installation path is identified
    - Expected local URLs or endpoints are documented


## Testing Backlog

- [x] P0 / M — Add feature engineering tests
  - Description: Verify that raw inputs are transformed into the expected model-ready features.
  - Acceptance criteria:
    - Core engineered features have test coverage
    - Mod parsing behavior is tested
    - Edge cases for missing or unusual values are identified

- [x] P0 / S — Add model loading tests
  - Description: Confirm the service can load serialized artifacts as expected.
  - Acceptance criteria:
    - Happy-path model load is tested
    - Missing-file behavior is tested or explicitly handled
    - Artifact path assumptions are covered

- [x] P0 / S — Add API health test
  - Description: Ensure the service reports health correctly when started locally.
  - Acceptance criteria:
    - `/health` returns the expected status code
    - Response shape is asserted
    - Failure expectations are documented if models are unavailable

- [x] P0 / M — Add API prediction test
  - Description: Verify that `/predict` accepts valid input and returns the expected contract.
  - Acceptance criteria:
    - A valid sample request is tested
    - Output fields are asserted
    - Numeric prediction values are type-checked

- [x] P0 / S — Add invalid input test
  - Description: Verify that malformed or incomplete requests fail predictably.
  - Acceptance criteria:
    - Missing required fields are rejected
    - Invalid field types are rejected
    - The error response remains understandable


## Documentation Backlog

- [x] P0 / M — Write README backlog item
  - Description: Plan a README that explains the project clearly to recruiters or collaborators.
  - Acceptance criteria:
    - Project purpose is covered
    - Setup steps are covered
    - ML + API scope is summarized

- [x] P0 / S — Write setup instructions backlog item
  - Description: Define the minimum local setup steps for reproducible development.
  - Acceptance criteria:
    - Python environment expectations are documented
    - Dependency installation approach is documented
    - Local development prerequisites are listed

- [x] P0 / S — Write training instructions backlog item
  - Description: Document how a user should generate model artifacts locally.
  - Acceptance criteria:
    - Training entry point is identified
    - Input data expectations are documented
    - Output artifact locations are documented

- [x] P1 / S — Add API usage example backlog item
  - Description: Plan a small request/response example for local users.
  - Acceptance criteria:
    - Example request is included
    - Example response is included
    - Local invocation method is documented

- [x] P0 / S — Write short model card backlog item
  - Description: Plan a concise model card describing data, metrics, and limitations.
  - Acceptance criteria:
    - Intended use is documented
    - Evaluation summary is planned
    - Key limitations are listed

- [x] P0 / S — Document assumptions and limitations
  - Description: Capture the boundaries of the MVP so the project remains honest and understandable.
  - Acceptance criteria:
    - Data assumptions are documented
    - Modeling limitations are documented
    - API limitations are documented


## Definition of Done

- [x] scikit-learn training pipeline exists
- [x] serialized model artifact exists
- [x] FastAPI prediction endpoint works
- [x] tests pass
- [x] short model card exists
- [x] service trains locally and serves predictions
- [x] assumptions and limitations are documented

## Suggested Repository Structure

```text
osu-skill-predictor/
  app/
    __init__.py
    main.py
    schemas.py
    predict.py

  ml/
    __init__.py
    train.py
    features.py
    evaluate.py
    pipeline.py

  notebooks/
    00_backlog.ipynb
    01_data_exploration.ipynb
    02_baseline_model.ipynb

  data/
    raw/
    processed/
    sample/

  models/
    pass_model.joblib
    accuracy_model.joblib

  tests/
    test_api.py
    test_features.py
    test_model_loading.py

  docs/
    dataset_schema.md
    api_usage.md

  model_card.md
  README.md
  requirements.txt
  pyproject.toml
```

## Milestones

### M0: project skeleton

- Establish the repository layout.
- Create the notebook-first planning artifacts.
- Lock the project name and scope around `osu-skill-predictor`.

### M1: dataset schema and sample data

- Define the dataset schema.
- Create a small local sample dataset.
- Document provenance and constraints.

### M2: baseline model

- Implement first-pass feature engineering.
- Train the first classifier and regressor.
- Evaluate simple baseline metrics.

### M3: serialized training pipeline

- Formalize preprocessing with scikit-learn pipelines.
- Serialize inference-ready artifacts with joblib.
- Confirm training outputs are reproducible locally.

### M4: FastAPI prediction API

- Add request/response schemas.
- Add `/health` and `/predict`.
- Load serialized models from the service.

### M5: tests and documentation

- Add automated tests for features, model loading, and API endpoints.
- Write README, setup, training, and usage docs.
- Add the short model card.

### M6: portfolio polish

- Improve naming, docs, and examples.
- Tighten assumptions and limitation notes.
- Make the project easy to review on GitHub.

## Task Format

Each task in this backlog should follow this structure:

```md
- [ ] P0 / M — Create initial dataset schema
  - Description: Define the columns required for the first training dataset.
  - Acceptance criteria:
    - `docs/dataset_schema.md` exists
    - Required input features are documented
    - Targets are documented
```

Required parts for every task:

- Checkbox.
- Priority: `P0`, `P1`, or `P2`.
- Estimated effort: `S`, `M`, or `L`.
- Short description.
- Acceptance criteria.

## How to use this backlog

- Complete tasks milestone by milestone.
- Keep the notebook updated as decisions change.
- Mark tasks as done when the acceptance criteria are met.
- Add notes under each milestone as implementation progresses.
- Avoid jumping to advanced features before finishing the MVP.

Recommended working order:

1. Finish M0 and M1 before writing production code.
2. Build the simplest valid baseline in M2.
3. Only then formalize serialization, API, tests, and polish.

In [ ]:
# Optional scratchpad for milestone notes or task tracking
